In [1]:
import os
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

import constants as k
import utils
from calculate_instantaneous_firing_rate import calculate_firing_rates

In [2]:
data_dir = '/Users/rebekahzhang/data/neural_data'
pickle_dir = Path(os.path.join(data_dir, 'session_pickles'))
figure_dir = os.path.join(data_dir, 'figures')

units_vetted = pd.read_csv(os.path.join(data_dir, 'units_vetted.csv'), index_col=0).sort_values('unit_id')
sessions_vetted = pd.read_csv(os.path.join(data_dir, 'sessions_vetted.csv'), index_col=0).sort_values('id')

In [3]:
sessions_vetted

,date,mouse,insertion_number,region,potential problems,sorting notes,First_X_Column,datetime,paramset_idx,num_units,id,length,num_trials,num_missed,wait_length_mean
0,2024-07-13,RZ034,1,str,NaN,should be all good now,Done,2024-07-13 12:58:26,101,47,RZ034_2024-07-13_str,2443.503482,328,0,3.317166
1,2024-07-14,RZ034,1,str,NaN,NaN,Done,2024-07-14 12:52:46,101,31,RZ034_2024-07-14_str,2832.875992,408,0,2.804475
3,2024-07-12,RZ036,1,str,NaN,NaN,Done,2024-07-12 12:50:31,101,45,RZ036_2024-07-12_str,3274.883860,299,3,6.628682
2,2024-07-12,RZ036,0,v1,NaN,NaN,Done,2024-07-12 12:50:31,101,15,RZ036_2024-07-12_v1,3274.883860,299,3,6.628682
4,2024-07-13,RZ036,1,str,NaN,NaN,Done,2024-07-13 14:29:03,101,30,RZ036_2024-07-13_str,4400.303239,295,15,10.838219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,2025-02-20,RZ065,0,v1,NaN,missing cluster_si_unit_id.tsv. curation delet...,Done,2025-02-20 12:02:05,101,57,RZ065_2025-02-20_v1,3086.800761,217,0,5.559549
91,2025-02-21,RZ065,1,str,NaN,NaN,Done,2025-02-21 11:15:15,101,19,RZ065_2025-02-21_str,3731.853914,302,0,3.075755
92,2025-02-21,RZ065,0,v1,NaN,"no bug, new phy",Done,2025-02-21 11:15:15,101,27,RZ065_2025-02-21_v1,3731.853914,302,0,3.075755
90,2025-02-22,RZ065,1,str,NaN,NaN,Done,2025-02-22 13:03:06,101,162,RZ065_2025-02-22_str,5265.110723,365,8,4.910904


In [4]:
session_id='RZ051_2024-11-20_v1'
unit_id=111
events, trials, spikes, session_fr = utils.get_data_for_debugging(pickle_dir, units_vetted, 
                                                                  session_id, unit_id)

In [5]:
def prepare_data_for_bg_fr_with_quantiles(trials, spikes, sorter, num_qantile):
    num_qantile = 5
    trials_no_miss = trials[trials.missed == False].copy()
    trials_no_miss['quantile'] = pd.qcut(trials[sorter], q=num_qantile, labels=list(range(1, num_qantile + 1)))

    bg_spikes = spikes[spikes['period'] == 'background']
    spike_counts = bg_spikes.groupby('trial_id').size()

    trials_no_miss = trials[trials.missed == False].copy()
    trials_no_miss['quantile'] = pd.qcut(trials[sorter], q=num_qantile, labels=list(range(1, num_qantile + 1)))
    trials_no_miss['bg_spike_count'] = trials_no_miss['trial_id'].map(spike_counts).fillna(0)
    trials_no_miss['bg_firing_rate'] = trials_no_miss['bg_spike_count'] / trials_no_miss['background_length']
    trials_bg_fr = trials_no_miss[['trial_id', 'quantile', 'wait_length', 'bg_spike_count', 'bg_firing_rate']].copy()
    return trials_bg_fr

In [ ]:
def bg_fr_regression(trials_bg_fr):
    tw_list = []
    fr_list = []
    trials_by_quantile = trials_bg_fr.groupby('quantile', observed=False)
    for _, group in trials_by_quantile:
        tw_mean = group['wait_length'].mean()
        tw_list.append(tw_mean)
        fr_mean = group['bg_firing_rate'].mean()
        fr_list.append(fr_mean)

    # Convert lists to numpy arrays
    x = np.array(tw_list)
    y = np.array(fr_list)

    # Add constant for intercept
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    unit_bg_fr = {
        "x": x,
        "y": y,
        "slope": model.params[1],
        "intercept": model.params[0],
        "r_squared": model.rsquared,
        "p_value": model.pvalues[1],
        "n_quantiles": len(x),
        "significant": model.pvalues[1] < 0.05
    }
    return unit_bg_fr

In [ ]:
sorter = k.WAIT_LENGTH

In [ ]:
units_by_session = units_vetted.groupby("session_id")
unit_data_list = []
for session_id, session_units in units_by_session:
    events, trials, units = utils.get_session_data(session_id, pickle_dir)
    region = session_units['region'].iloc[0]
    for _, unit_info in session_units.iterrows():
        unit_id = unit_info['unit_id']
        spikes = units[unit_info['id']]
        trials_bg_fr = prepare_data_for_bg_fr_with_quantiles(trials, spikes, sorter, num_qantile=5)
        unit_bg_fr = bg_fr_regression(trials_bg_fr)
        unit_data = {'unit_id': unit_id, 'region': region} | unit_bg_fr
        unit_data_list.append(unit_data)
unit_bg_fr_df = pd.DataFrame(unit_data_list)

In [ ]:
unit_bg_fr_df.to_csv(os.path.join(data_dir, 'unit_bg_fr.csv'))

In [ ]:
bg_sig_units = unit_bg_fr_df[unit_bg_fr_df.significant==True]

In [ ]:
decision_units_by_region = bg_sig_units.groupby("region")
for region, region_data in decision_units_by_region:
    source_folder = os.path.join(figure_dir, 'raster_histo_to_cue_on_by_wait_length', region)
    destination_folder = os.path.join(figure_dir, 'bg_fr_diffferent_sig', region)
    
    # Create destination folder if it doesn't exist
    os.makedirs(destination_folder, exist_ok=True)
    
    print(f"Processing region: {region}")
    print(f"Destination folder: {destination_folder}")
    
    for _, unit_data in region_data.iterrows():
        unit_id = unit_data['unit_id']
        session_id = unit_data['unit_id']  # Make sure this column exists
        
        source_path = os.path.join(source_folder, f"{session_id}.png")
        destination_path = os.path.join(destination_folder, f"{session_id}.png")
        
        try:
            shutil.copy2(source_path, destination_path)  # copy2 preserves metadata
        except FileNotFoundError:
            print(f"Warning: Source file not found - {source_path}")
        except Exception as e:
            print(f"Error copying {source_path}: {str(e)}")

In [ ]:
tw_list = []
fr_list = []
trials_by_quantile = trials_bg_fr.groupby('quantile', observed=False)
for quantile, group in trials_by_quantile:
    tw_mean = group['wait_length'].mean()
    tw_list.append(tw_mean)
    fr_mean = group['bg_firing_rate'].mean()
    fr_list.append(fr_mean)

In [ ]:
# Convert lists to numpy arrays
x = np.array(tw_list)
y = np.array(fr_list)

# Add constant for intercept
X = sm.add_constant(x)
model = sm.OLS(y, X).fit()

slope = model.params[1]
intercept = model.params[0]
r_squared = model.rsquared
p_value = model.pvalues[1]

print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R^2: {r_squared}")
print(f"P-value: {p_value}")

In [ ]:
fr_list

old code below

In [ ]:
def prepare_data_for_bg_fr(trials, spikes, num_qantile, sorter=sorter):
    trials = trials[trials.missed == False].copy()
    trials['quantile'] = pd.qcut(trials[sorter], q=num_qantile, labels=list(range(1,num_qantile+1)))

    trials['aligned_start_time'] = trials['cue_on_time'] - trials['cue_off_time']
    trials['aligned_end_time'] = trials['cue_off_time'] - trials['cue_off_time']

    spikes_bg = spikes.loc[spikes.period == k.BACKGROUND]
    common_trial_ids = pd.merge(trials[['trial_id']], spikes_bg[['trial_id']], 
                                    on='trial_id')['trial_id'].unique().tolist()
        
    trials_histo = trials.loc[trials.trial_id.isin(common_trial_ids)]
    spikes_histo = spikes_bg.loc[spikes.trial_id.isin(common_trial_ids)]

    return trials_histo, spikes_histo

In [ ]:
def generate_10_quantile_linear_regression_data(trials_histo, spikes_histo, anchor, time_step, trial_count_mask, sigma, unit_id):
    # Calculate for each quantile
    x = []
    y = []
    for quantile in trials_histo['quantile'].dropna().unique():
        trials_quantile = trials_histo.loc[trials_histo['quantile'] == quantile]
        spikes_quantile = spikes_histo.loc[spikes_histo.trial_id.isin(trials_quantile.trial_id)]
        bin_centers_q, mean_fr_q, _ = calculate_firing_rates(
            trials_quantile, spikes_quantile, anchor, time_step, trial_count_mask, sigma
        )
        
        # Find indices within the time window for this quantile
        time_window_mask_q = (bin_centers_q >= -0.5) & (bin_centers_q <= 0)
        mean_fr_q_window = mean_fr_q[time_window_mask_q]
        
        # Calculate window metrics for quantile
        if len(mean_fr_q_window) > 0:
            q_window_mean_fr = mean_fr_q_window.mean()
        else:
            q_window_mean_fr = np.nan
        
        x.append(trials_quantile['wait_length'].mean())
        y.append(q_window_mean_fr)

    # Convert to numpy arrays and add constant term
    x = np.array(x)
    y = np.array(y)

    # Filter out any NaN values that might be present
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    # Only perform regression if we have at least 2 valid points
    if len(x_valid) >= 2:
        # Add constant term for intercept
        X = sm.add_constant(x_valid)
        model = sm.OLS(y_valid, X).fit()
        
        # Extract parameters correctly
        slope = model.params[1]  # Note: [1] is the slope
        intercept = model.params[0]  # [0] is the intercept
        r_squared = model.rsquared
        p_value = model.pvalues[1]  # p-value for the slope
        
        quantile_data_dict = {
            "x": x_valid,
            "y": y_valid,
            "slope": slope,
            "intercept": intercept,
            "r_squared": r_squared,
            "p_value": p_value,
            "n_quantiles": len(x_valid),
            "significant": p_value < 0.05  # Added significance flag
        }
    else:
        # Return NaN values if not enough data points
        quantile_data_dict = {
            "x": x_valid,
            "y": y_valid,
            "slope": np.nan,
            "intercept": np.nan,
            "r_squared": np.nan,
            "p_value": np.nan,
            "n_quantiles": len(x_valid),
            "significant": False
            }
    return quantile_data_dict

In [ ]:
anchor = k.TO_DECISION
sorter = k.WAIT_LENGTH

time_step = 0.1
trial_count_mask = 3
sigma = 1

unit_id = 'RZ050_2024-11-19_v1_unit-19'

trials_histo, spikes_histo = prepare_data_for_bg_fr(trials, spikes, num_qantile=5, sorter=sorter)
quantile_dict = generate_10_quantile_linear_regression_data(
    trials_histo, spikes_histo, anchor, time_step, trial_count_mask, sigma, unit_id
    )

In [ ]:
quantile_dict

In [ ]:
units_by_session = units_vetted.groupby("session_id")
unit_ten_quantile_data_list = []
for session_id, session_units in units_by_session:
    events, trials, units = utils.get_session_data(session_id, pickle_dir)
    region = session_units['region'].iloc[0]
    for _, unit_info in session_units.iterrows():
        unit_id = unit_info['unit_id']
        spikes = units[unit_info['id']]
        trials_histo, spikes_histo = prepare_data_for_bg_fr(trials, spikes, 5, sorter)
        unit_ten_quantile_data_dict = generate_10_quantile_linear_regression_data(
            trials_histo, spikes_histo, anchor, time_step, trial_count_mask, sigma, unit_id
            )
        unit_ten_quantile_data = {'unit_id': unit_id, 'region': region} | unit_ten_quantile_data_dict
        unit_ten_quantile_data_list.append(unit_ten_quantile_data)
unit_ten_quantile_data_df = pd.DataFrame(unit_ten_quantile_data_list)

In [ ]:
unit_ten_quantile_data_df

In [ ]:
significant_decision_units = unit_ten_quantile_data_df[unit_ten_quantile_data_df['significant'] == True]

In [ ]:
significant_decision_units